# Face swap: servidor FaceFusion + Gradio com registro automático

Antes de rodar:
1. Runtime → Alterar tipo de ambiente → **T4 GPU**.
2. Crie os segredos no 🔑 (barra lateral) e ative "Acesso ao notebook": `API_TOKEN` e `REGISTER_SECRET`.
3. Na célula 3, troque `SEU-PROJETO` pelo domínio do seu projeto no Vercel.

Rode as células em ordem. Quando a célula 5 mostrar `[heartbeat] ... registro: ok`, o site já está conectado.

In [ ]:
!nvidia-smi

## Célula 2: instalar o FaceFusion (5 a 10 min)

In [ ]:
!apt-get install -y -q ffmpeg > /dev/null
!git clone --depth 1 https://github.com/facefusion/facefusion.git /content/facefusion
%cd /content/facefusion
!python install.py --onnxruntime cuda --skip-conda
%cd /content

## Célula 3: carregar segredos e definir o domínio do site

In [ ]:
import os
from google.colab import userdata

os.environ["API_TOKEN"] = userdata.get("API_TOKEN")
os.environ["REGISTER_SECRET"] = userdata.get("REGISTER_SECRET")
os.environ["SITE_URL"] = "https://SEU-PROJETO.vercel.app"  # troque pelo seu domínio do Vercel, sem barra no final
print("Segredos carregados.")

## Célula 4: criar o servidor Gradio com heartbeat

In [ ]:
%%writefile /content/server.py
import os, sys, time, shutil, uuid, subprocess, threading
from pathlib import Path
import requests
import gradio as gr

FF_DIR = Path("/content/facefusion")
JOBS = Path("/content/jobs")
JOBS.mkdir(parents=True, exist_ok=True)
TOKEN = os.environ["API_TOKEN"]
SITE = os.environ["SITE_URL"].rstrip("/")
REGISTER_SECRET = os.environ["REGISTER_SECRET"]
HEARTBEAT_SECONDS = 120

def cleanup_old_jobs(max_age=2 * 3600):
    now = time.time()
    for d in JOBS.iterdir():
        if now - d.stat().st_mtime > max_age:
            shutil.rmtree(d, ignore_errors=True)

def swap(video, photo, token):
    if token != TOKEN:
        raise gr.Error("Token inválido.")
    if not video or not photo:
        raise gr.Error("Envie um vídeo e uma foto.")

    cleanup_old_jobs()
    job = JOBS / uuid.uuid4().hex
    job.mkdir()
    out = job / "output.mp4"

    cmd = [
        sys.executable, "facefusion.py", "headless-run",
        "--source-paths", photo,
        "--target-path", video,
        "--output-path", str(out),
        "--processors", "face_swapper",
        "--execution-providers", "cuda",
    ]
    try:
        proc = subprocess.run(cmd, cwd=FF_DIR, capture_output=True, text=True, timeout=1500)
    except subprocess.TimeoutExpired:
        raise gr.Error("Tempo limite excedido.")

    if proc.returncode != 0 or not out.exists():
        print("STDOUT:", proc.stdout[-3000:])
        print("STDERR:", proc.stderr[-3000:])
        raise gr.Error("O FaceFusion falhou. Veja o log no Colab.")
    return str(out)

def register(share_url):
    r = requests.post(
        f"{SITE}/api/colab/register",
        headers={"Authorization": f"Bearer {REGISTER_SECRET}"},
        json={"url": share_url},
        timeout=20,
    )
    return r.status_code == 200, r.status_code

def heartbeat(share_url):
    last = None
    while True:
        try:
            ok, code = register(share_url)
            state = "ok" if ok else f"erro HTTP {code}"
        except Exception as e:
            state = f"falhou ({e})"
        if state != last:
            print(f"[heartbeat] {time.strftime('%H:%M:%S')} registro: {state}")
            last = state
        time.sleep(HEARTBEAT_SECONDS)

with gr.Blocks(title="FaceFusion API") as demo:
    gr.Markdown("API de face swap usada pelo site.")
    v_in = gr.Video(label="Vídeo")
    p_in = gr.Image(type="filepath", label="Foto")
    t_in = gr.Textbox(label="Token", type="password")
    v_out = gr.Video(label="Resultado")
    btn = gr.Button("Processar")
    btn.click(swap, [v_in, p_in, t_in], v_out, api_name="swap", concurrency_limit=1)

demo.queue(default_concurrency_limit=1, max_size=5)
_, local_url, share_url = demo.launch(share=True, prevent_thread_lock=True)
if not share_url:
    raise SystemExit("O Gradio não gerou link público.")

print("URL pública:", share_url)
threading.Thread(target=heartbeat, args=(share_url,), daemon=True).start()
demo.block_thread()

## Célula 5: rodar o servidor (deixe esta célula executando e não feche a aba)

In [ ]:
!python /content/server.py